# Online Retail II — Exploration

This notebook profiles the raw Online Retail II dataset before cleaning. The goal is to identify data-quality issues and define a Stage 2 cleaning plan for customer analytics, including RFM segmentation, cohort retention, and churn modelling.

**Source:** UCI Machine Learning Repository — Online Retail II  
**Coverage:** December 2009 to December 2011  
**Raw File:** `data/raw/online_retail_II.xlsx` (two sheets, one per year)

In [1]:
import re

import pandas as pd

print("pandas version:", pd.__version__)


pandas version: 3.0.2


## Load data

The workbook contains two yearly sheets. Both are loaded explicitly and combined into one transaction-level DataFrame.


In [2]:
# Load each yearly sheet explicitly to avoid relying on workbook sheet order
RAW_PATH = "../data/raw/online_retail_II.xlsx"

df_2009 = pd.read_excel(RAW_PATH, sheet_name="Year 2009-2010")
df_2010 = pd.read_excel(RAW_PATH, sheet_name="Year 2010-2011")

# Combine both years into one transaction-level DataFrame
df = pd.concat([df_2009, df_2010], ignore_index=True)

INSPECTION_COLUMNS = df.columns.tolist()

print(f"Loaded {len(df):,} rows")


Loaded 1,067,371 rows


In [3]:
# Sanity check that concat preserved all rows and both sheets share the same schema
assert list(df_2009.columns) == list(df_2010.columns), "Sheet column mismatch"
assert len(df_2009) + len(df_2010) == len(df), "Row count mismatch after concat"

print(f"2009-2010 rows: {len(df_2009):,}")
print(f"2010-2011 rows: {len(df_2010):,}")
print(f"Combined rows:  {len(df):,}")

2009-2010 rows: 525,461
2010-2011 rows: 541,910
Combined rows:  1,067,371


## Basic dataset overview

In [4]:
# Check the number of rows and columns
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")


Rows: 1,067,371
Columns: 8


In [5]:
# Review column data types and non-null counts
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[us]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 65.1+ MB


In [6]:
# Preview the first few rows for a quick structure check
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [7]:
# Preview the last few rows to confirm the combined dataset ends correctly
df.tail()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
1067366,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France
1067367,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France
1067368,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France
1067369,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,2011-12-09 12:50:00,4.95,12680.0,France
1067370,581587,POST,POSTAGE,1,2011-12-09 12:50:00,18.00,12680.0,France


## Missing values

Missing values are checked before the column-level audit to identify the main data-completeness issues.


In [8]:
# Summarise missing values by column, sorted from highest to lowest percentage
missing = pd.DataFrame({
    "null_count": df.isna().sum(),
    "null_pct": (df.isna().mean() * 100).round(2)
})

missing.sort_values("null_pct", ascending=False)

,null_count,null_pct
Customer ID,243007,22.77
Description,4382,0.41
StockCode,0,0.00
Invoice,0,0.00
Quantity,0,0.00
InvoiceDate,0,0.00
Price,0,0.00
Country,0,0.00


**Missing values summary**

Two columns have missing values: `Customer ID` (243,007 rows, 22.77%) and `Description` (4,382 rows, 0.41%). The other six columns are fully populated. Both are examined in detail in their column audit sections; the missing-customer rows are the main filter for Stage 2 cleaning.

## Column-level data quality audit

Each column is checked for missing values, data type, unusual values, format issues, and cleaning decisions needed for Stage 2.


### Invoice

In [9]:
# Profile Invoice for missing values, data type, and cardinality
print(f"Null count:    {df['Invoice'].isna().sum():,}")
print(f"Dtype:         {df['Invoice'].dtype}")
print(f"Unique values: {df['Invoice'].nunique():,}")

Null count:    0
Dtype:         object
Unique values: 53,628


In [10]:
# Check which Invoice values match the UCI format: 6 digits, optionally prefixed with C
invoice_text = df["Invoice"].astype(str).str.strip()

normal_invoice = (
    invoice_text.str.len().eq(6)
    & invoice_text.str.isdigit()
)

cancellation_invoice = (
    invoice_text.str.startswith("C")
    & invoice_text.str[1:].str.len().eq(6)
    & invoice_text.str[1:].str.isdigit()
)

conforming = normal_invoice | cancellation_invoice

print(f"Conforming:     {conforming.sum():,} ({conforming.mean():.2%})")
print(f"Non-conforming: {(~conforming).sum():,} ({(~conforming).mean():.2%})")

Conforming:     1,067,365 (100.00%)
Non-conforming: 6 (0.00%)


In [11]:
# Inspect non-conforming Invoice rows to identify non-standard invoice types
df.loc[
    ~conforming,
    ["Invoice", "StockCode", "Description", "Quantity", "Price", "Customer ID"]
]

,Invoice,StockCode,Description,Quantity,Price,Customer ID
179403,A506401,B,Adjust bad debt,1,-53594.36,NaN
276274,A516228,B,Adjust bad debt,1,-44031.79,NaN
403472,A528059,B,Adjust bad debt,1,-38925.87,NaN
825443,A563185,B,Adjust bad debt,1,11062.06,NaN
825444,A563186,B,Adjust bad debt,1,-11062.06,NaN
825445,A563187,B,Adjust bad debt,1,-11062.06,NaN


**Invoice audit summary**

The non-conforming `Invoice` rows are `A`-prefix bad-debt adjustments with missing `Customer ID`. They are not customer-level purchase records and will be removed during Stage 2 when missing-customer rows and non-product records are filtered.


### StockCode

In [12]:
# Profile StockCode for missing values, data type, and cardinality
print(f"Null count:    {df['StockCode'].isna().sum():,}")
print(f"Dtype:         {df['StockCode'].dtype}")
print(f"Unique values: {df['StockCode'].nunique():,}")

Null count:    0
Dtype:         object
Unique values: 5,305


In [13]:
# Check StockCode values against the common product-code pattern
sc_text = df["StockCode"].astype(str)

five_digit = (
    sc_text.str.len().eq(5)
    & sc_text.str.isdigit()
)

five_digit_with_letter = (
    sc_text.str.len().eq(6)
    & sc_text.str[:5].str.isdigit()
    & sc_text.str[5:].str.isalpha()
    & sc_text.str[5:].str.isupper()
)

sc_conforming = five_digit | five_digit_with_letter

print(f"Conforming:     {sc_conforming.sum():,} ({sc_conforming.mean():.2%})")
print(f"Non-conforming: {(~sc_conforming).sum():,} ({(~sc_conforming).mean():.2%})")

print(f"\nUnique conforming codes:     {sc_text[sc_conforming].nunique():,}")
print(f"Unique non-conforming codes: {sc_text[~sc_conforming].nunique():,}")

Conforming:     1,056,632 (98.99%)
Non-conforming: 10,739 (1.01%)

Unique conforming codes:     5,065
Unique non-conforming codes: 240


In [14]:
# StockCode audit — check leading and trailing whitespace
stockcode_text = df["StockCode"].astype(str)

print(f"Leading whitespace:  {stockcode_text.str.startswith(' ').sum():,}")
print(f"Trailing whitespace: {stockcode_text.str.endswith(' ').sum():,}")

Leading whitespace:  0
Trailing whitespace: 1


In [15]:
# Count the most common non-conforming StockCode values
non_conforming_codes = sc_text[~sc_conforming].value_counts()

print(f"Total unique non-conforming codes: {len(non_conforming_codes):,}")
print("\nTop 25 by row count:")
print(non_conforming_codes.head(25))

Total unique non-conforming codes: 240

Top 25 by row count:
StockCode
POST            2122
DOT             1446
M               1421
15056BL          923
C2               282
79323LP          232
D                177
18098c           142
79323GR          123
72349b           121
15056n           121
47566b           116
S                104
84970s           103
BANK CHARGES     102
85123a           101
15056bl           93
84997b            90
82494l            88
84997d            86
84997a            81
84997c            80
84509a            75
47591d            74
84970l            71
Name: count, dtype: int64


In [16]:
# Inspect sample descriptions for the most common non-conforming StockCode values
top_codes = non_conforming_codes.head(25).index

print(f"{'StockCode':<18} {'Rows':>6}  Sample description")
print("-" * 80)

for code in top_codes:
    descs = df.loc[df["StockCode"] == code, "Description"].dropna()
    sample = descs.iloc[0] if len(descs) > 0 else "<no description>"
    print(f"{code:<18} {non_conforming_codes[code]:>6,}  {sample!r}")

StockCode            Rows  Sample description
--------------------------------------------------------------------------------
POST                2,122  'POSTAGE'
DOT                 1,446  'DOTCOM POSTAGE'
M                   1,421  'Manual'
15056BL               923  'EDWARDIAN PARASOL BLACK'
C2                    282  'CARRIAGE'
79323LP               232  'LIGHT PINK CHERRY LIGHTS'
D                     177  'Discount'
18098c                142  'PORCELAIN BUTTERFLY OIL BURNER'
79323GR               123  'GREEN CHERRY LIGHTS'
72349b                121  'SET/6 PURPLE BUTTERFLY T-LIGHTS'
15056n                121  'EDWARDIAN PARASOL NATURAL'
47566b                116  'TEA TIME PARTY BUNTING'
S                     104  'SAMPLES'
84970s                103  'HANGING HEART ZINC T-LIGHT HOLDER'
BANK CHARGES          102  ' Bank Charges'
85123a                101  'WHITE HANGING HEART T-LIGHT HOLDER'
15056bl                93  'EDWARDIAN PARASOL BLACK'
84997b                 90  'RED 3 PI

In [17]:
# StockCode audit — categorise non-conforming codes by structural pattern
def classify_stockcode(code):
    code = str(code).strip()

    if re.fullmatch(r"\d{5}[a-z]{1,2}", code):
        return "lowercase suffix product-code variant"

    if re.fullmatch(r"\d{5}[A-Z]{2}", code):
        return "two-letter product-code variant"

    if re.fullmatch(r"[A-Za-z]+", code):
        return "letters only"

    if re.fullmatch(r"[A-Z][0-9]+", code):
        return "single letter + digits"

    if re.fullmatch(r"[A-Z\s]+", code):
        return "uppercase phrase with spaces"

    if re.fullmatch(r"gift_\d+_\d+", code):
        return "gift voucher code"

    if re.match(r"^DCGS", code):
        return "DCGS-prefix code"

    return "other"


non_conf_df = pd.DataFrame({
    "StockCode": non_conforming_codes.index,
    "row_count": non_conforming_codes.values,
})

non_conf_df["category"] = non_conf_df["StockCode"].apply(classify_stockcode)

print("Non-conforming StockCode breakdown by structural pattern:\n")

summary = (
    non_conf_df
    .groupby("category")
    .agg(
        unique_codes=("StockCode", "count"),
        total_rows=("row_count", "sum"),
    )
    .sort_values("total_rows", ascending=False)
)

print(summary)

Non-conforming StockCode breakdown by structural pattern:

                                       unique_codes  total_rows
category                                                       
letters only                                     16        5477
lowercase suffix product-code variant           173        3366
two-letter product-code variant                   4        1279
single letter + digits                            2         283
DCGS-prefix code                                 30         108
uppercase phrase with spaces                      1         102
gift voucher code                                 9         100
other                                             5          24


In [18]:
# StockCode audit — inspect all letters-only codes
letters_only_codes = (
    non_conf_df
    .loc[non_conf_df["category"] == "letters only", "StockCode"]
    .tolist()
)

print(f"Letters-only codes: {len(letters_only_codes)}\n")

print(
    f"{'StockCode':<15} {'Rows':>6} "
    f"{'Missing CustID':>14} {'Valid CustID':>12}  Sample description"
)
print("-" * 105)

for code in letters_only_codes:
    rows = df.loc[df["StockCode"] == code]
    descriptions = rows["Description"].dropna()

    sample_description = (
        descriptions.iloc[0]
        if len(descriptions) > 0
        else "<no description>"
    )

    print(
        f"{code:<15} {len(rows):>6,} "
        f"{rows['Customer ID'].isna().sum():>14,} "
        f"{rows['Customer ID'].notna().sum():>12,}  "
        f"{sample_description!r}"
    )

Letters-only codes: 16

StockCode         Rows Missing CustID Valid CustID  Sample description
---------------------------------------------------------------------------------------------------------
POST             2,122            103        2,019  'POSTAGE'
DOT              1,446          1,430           16  'DOTCOM POSTAGE'
M                1,421            306        1,115  'Manual'
D                  177              3          174  'Discount'
S                  104            104            0  'SAMPLES'
ADJUST              67              6           61  'Adjustment by john on 26/01/2010 16'
AMAZONFEE           43             43            0  'AMAZON FEE'
DCGSSGIRL           25             25            0  'update'
DCGSSBOY            23             23            0  'update'
PADS                19              0           19  'PADS TO MATCH ALL CUSHIONS'
CRUK                16              0           16  'CRUK Commission'
B                    6              6            0  'A

In [19]:
# StockCode audit — verify whether PADS behaves like a product
pads_rows = df.loc[df["StockCode"] == "PADS"]

print("PADS Quantity and Price summary:")
print(pads_rows[["Quantity", "Price"]].describe())

print("\nAll PADS rows:")
print(pads_rows[INSPECTION_COLUMNS].to_string(index=False))

PADS Quantity and Price summary:
        Quantity      Price
count  19.000000  19.000000
mean    0.894737   1.927211
std     0.458831   8.396399
min    -1.000000   0.000000
25%     1.000000   0.001000
50%     1.000000   0.001000
75%     1.000000   0.001000
max     1.000000  36.600000

All PADS rows:
Invoice StockCode                Description  Quantity         InvoiceDate  Price  Customer ID        Country
 494914      PADS PADS TO MATCH ALL CUSHIONS         1 2010-01-19 17:04:00  0.001      16705.0 United Kingdom
 496222      PADS PADS TO MATCH ALL CUSHIONS         1 2010-01-29 13:53:00  0.001      13583.0 United Kingdom
 496473      PADS PADS TO MATCH ALL CUSHIONS         1 2010-02-01 15:38:00  0.001      17350.0 United Kingdom
 496643      PADS PADS TO MATCH ALL CUSHIONS         1 2010-02-03 11:58:00  0.001      13408.0 United Kingdom
 497935      PADS PADS TO MATCH ALL CUSHIONS         1 2010-02-15 10:47:00  0.001      13408.0 United Kingdom
 498562      PADS PADS TO MATCH ALL CUS

In [20]:
# StockCode audit — inspect codes that did not match any structural pattern
other_codes = (
    non_conf_df
    .loc[non_conf_df["category"] == "other", "StockCode"]
    .tolist()
)

print(f"Codes in 'other': {other_codes}\n")

print(f"{'StockCode':<18} {'Rows':>6}  Sample description")
print("-" * 80)

for code in other_codes:
    rows = df.loc[df["StockCode"] == code]
    descriptions = rows["Description"].dropna()

    sample_description = (
        descriptions.iloc[0]
        if len(descriptions) > 0
        else "<no description>"
    )

    print(f"{code:<18} {len(rows):>6,}  {sample_description!r}")

Codes in 'other': ['TEST001', 'ADJUST2', 'SP1002', 'TEST002', '47503J ']

StockCode            Rows  Sample description
--------------------------------------------------------------------------------
TEST001                15  'This is a test product.'
ADJUST2                 3  'Adjustment by Peter on Jun 25 2010 '
SP1002                  3  "KID'S CHALKBOARD/EASEL"
TEST002                 2  'This is a test product.'
47503J                  1  'SET/3 FLORAL GARDEN TOOLS IN BAG'


In [21]:
# StockCode audit — inspect DCGS-prefix codes and missing descriptions
dcgs_codes = (
    non_conf_df
    .loc[non_conf_df["category"] == "DCGS-prefix code", "StockCode"]
    .tolist()
)

dcgs_breakdown = []
for code in dcgs_codes:
    rows = df.loc[df["StockCode"] == code]
    descriptions = rows["Description"].dropna()
    sample_description = (
        descriptions.iloc[0]
        if len(descriptions) > 0
        else None
    )
    dcgs_breakdown.append({
        "StockCode": code,
        "rows": len(rows),
        "rows_missing_description": rows["Description"].isna().sum(),
        "sample_description": sample_description,
    })

dcgs_summary = (
    pd.DataFrame(dcgs_breakdown)
    .sort_values("rows", ascending=False)
)

with pd.option_context("display.max_rows", None, "display.max_colwidth", 60):
    print(dcgs_summary.to_string(index=False))

print(f"\nDCGS codes total: {len(dcgs_codes)}")
print(f"DCGS rows total: {dcgs_summary['rows'].sum()}")
print(
    "DCGS rows with missing Description: "
    f"{dcgs_summary['rows_missing_description'].sum()}"
)

StockCode  rows  rows_missing_description               sample_description
 DCGS0058    31                         1                 MISO PRETTY  GUM
 DCGS0076    15                         0     SUNJAR LED NIGHT NIGHT LIGHT
 DCGS0003    14                         0              BOXED GLASS ASHTRAY
 DCGS0069     6                         0            OOH LA LA DOGS COLLAR
 DCGS0004     5                         1       HAYNES CAMPER SHOULDER BAG
 DCGS0072     4                         1           CAT CAMOUFLAGUE COLLAR
DCGS0066N     4                         2          NAVY CUDDLES DOG HOODIE
 DCGS0068     3                         0                DOGS NIGHT COLLAR
 DCGS0070     3                         1            CAMOUFLAGE DOG COLLAR
 DCGS0062     2                         1          ROAD-RAGE CAR FRESHENER
 DCGS0037     2                         1               KEY-RING CORKSCREW
 DCGS0044     1                         0          HANDZ-OFF CAR FRESHENER
 DCGS0006     1          

In [22]:
# StockCode audit — inspect gift voucher StockCodes
gift_codes = (
    non_conf_df
    .loc[non_conf_df["category"] == "gift voucher code", "StockCode"]
    .tolist()
)

print(f"Gift voucher codes: {len(gift_codes)}\n")

print(f"{'StockCode':<20} {'Rows':>6}  Sample description")
print("-" * 80)

for code in gift_codes:
    rows = df.loc[df["StockCode"] == code]
    descriptions = rows["Description"].dropna()

    sample_description = (
        descriptions.iloc[0]
        if len(descriptions) > 0
        else "<no description>"
    )

    print(f"{code:<20} {len(rows):>6,}  {sample_description!r}")

Gift voucher codes: 9

StockCode              Rows  Sample description
--------------------------------------------------------------------------------
gift_0001_20             29  'Dotcomgiftshop Gift Voucher £20.00'
gift_0001_30             29  'Dotcomgiftshop Gift Voucher £30.00'
gift_0001_10             16  'Dotcomgiftshop Gift Voucher £10.00'
gift_0001_50              8  'Dotcomgiftshop Gift Voucher £50.00'
gift_0001_40              7  'Dotcomgiftshop Gift Voucher £40.00'
gift_0001_80              4  'Dotcomgiftshop Gift Voucher £80.00'
gift_0001_70              3  'Dotcomgiftshop Gift Voucher £70.00'
gift_0001_60              2  '<no description>'
gift_0001_90              2  '<no description>'


In [23]:
# StockCode audit — check relationship between non-conforming StockCodes and Customer ID
category_lookup = non_conf_df.set_index("StockCode")["category"]

nc_rows = df.loc[~sc_conforming].copy()
nc_rows["stockcode_category"] = nc_rows["StockCode"].map(category_lookup)

print(f"Non-conforming StockCode rows: {len(nc_rows):,}")
print(f"  with missing Customer ID: {nc_rows['Customer ID'].isna().sum():,}")
print(f"  with valid Customer ID:   {nc_rows['Customer ID'].notna().sum():,}")

print("\nBreakdown by structural category:")

stockcode_customer_summary = (
    nc_rows
    .groupby("stockcode_category")["Customer ID"]
    .agg(
        rows="size",
        missing_customer_id=lambda s: s.isna().sum(),
        valid_customer_id=lambda s: s.notna().sum(),
    )
    .sort_values("rows", ascending=False)
)

stockcode_customer_summary["pct_missing_customer_id"] = (
    stockcode_customer_summary["missing_customer_id"]
    / stockcode_customer_summary["rows"]
    * 100
).round(1)

print(stockcode_customer_summary)

Non-conforming StockCode rows: 10,739


  with missing Customer ID: 5,819
  with valid Customer ID:   4,920

Breakdown by structural category:
                                       rows  missing_customer_id  \
stockcode_category                                                 
letters only                           5477                 2057   
lowercase suffix product-code variant  3366                 3366   
two-letter product-code variant        1279                   97   
single letter + digits                  283                   24   
DCGS-prefix code                        108                  108   
uppercase phrase with spaces            102                   64   
gift voucher code                       100                  100   
other                                    24                    3   

                                       valid_customer_id  \
stockcode_category                                         
letters only                                        3420   
lowercase suffix product-code varia

**StockCode audit summary**

`StockCode` is fully populated and mostly follows the common product-code pattern. The non-conforming values are mixed: some are valid product variants, while others are service, manual, voucher, test, adjustment, or system-style rows. One row has trailing whitespace (`47503J ` in the `other` category), justifying a Stage 2 strip on this column.

`PADS` was inspected separately because it has valid customer rows and a product-like description. However, almost all positive rows have a placeholder-like price of `0.001`, so it should be removed from the customer analytics dataset as a non-standard product-purchase row.

For Stage 2, strip whitespace from `StockCode` and treat the following 17 codes as non-product codes: `POST`, `DOT`, `M`, `m`, `D`, `S`, `ADJUST`, `AMAZONFEE`, `CRUK`, `B`, `GIFT`, `PADS`, `BANK CHARGES`, `C2`, `TEST001`, `TEST002`, and `ADJUST2`. The `gift voucher` codes, `DCGS-prefix` codes, the four DCGS letters-only codes (`DCGSSGIRL`, `DCGSSBOY`, `DCGSLBOY`, `DCGSLGIRL`), and the `lowercase suffix product-code variant` category are all 100% missing `Customer ID` and will be removed by the missing-customer filter.

### Description

In [24]:
# Description audit — basic profile
print(f"Null count:    {df['Description'].isna().sum():,}")
print(f"Null pct:      {df['Description'].isna().mean():.2%}")
print(f"Dtype:         {df['Description'].dtype}")
print(f"Unique values: {df['Description'].nunique():,}")

Null count:    4,382
Null pct:      0.41%
Dtype:         object
Unique values: 5,698


In [25]:
# Description audit — whitespace and casing profile
desc = df["Description"].dropna().astype(str)

leading_space = desc.str.startswith(" ").sum()
trailing_space = desc.str.endswith(" ").sum()
double_space = desc.str.contains("  ", regex=False).sum()

is_lowercase = desc.str.islower()
is_uppercase = desc.str.isupper()
neither_lower_nor_upper = (~is_lowercase & ~is_uppercase).sum()

print(f"Total non-null Description rows: {len(desc):,}\n")
print(f"  Leading whitespace:        {leading_space:,}")
print(f"  Trailing whitespace:       {trailing_space:,}")
print(f"  Double spaces inside:      {double_space:,}")
print()
print(f"  All lowercase:             {is_lowercase.sum():,}")
print(f"  All uppercase:             {is_uppercase.sum():,}")
print(f"  Neither lower nor upper:   {neither_lower_nor_upper:,}")

Total non-null Description rows: 1,062,989

  Leading whitespace:        2,624
  Trailing whitespace:       211,195
  Double spaces inside:      52,708

  All lowercase:             732
  All uppercase:             1,056,917
  Neither lower nor upper:   5,340


In [26]:
# Description audit — inspect lowercase and non-uppercase descriptions
desc = df["Description"].dropna().astype(str)

is_lowercase = desc.str.islower()
is_uppercase = desc.str.isupper()
neither_lower_nor_upper = ~is_lowercase & ~is_uppercase

print("Top lowercase descriptions:")
print(desc[is_lowercase].value_counts().head(25))

print("\nTop descriptions that are neither fully lowercase nor fully uppercase:")
print(desc[neither_lower_nor_upper].value_counts().head(25))

Top lowercase descriptions:
Description
check                    162
damages                   84
damaged                   81
found                     28
missing                   27
sold as set on dotcom     20
adjustment                16
dotcom                    12
amazon                    11
smashed                    9
thrown away                9
checked                    8
damages?                   7
mailout                    6
crushed                    6
given away                 6
counted                    5
wet damaged                5
ebay                       5
had been put aside         5
ebay sales                 4
temp                       4
broken                     3
test                       3
wet pallet                 3
Name: count, dtype: int64

Top descriptions that are neither fully lowercase nor fully uppercase:
Description
Manual                                 1426
BAG 250g SWIRLY MARBLES                 598
BAG 125g SWIRLY MARBLES              

In [27]:
# Description audit — inspect descriptions matching known suspicious terms
desc_text = df["Description"].dropna().astype(str)
desc_standard = desc_text.str.strip().str.upper()

known_suspicious_terms = [
    "MANUAL",
    "DISCOUNT",
    "BANK CHARGES",
    "CARRIAGE",
    "ADJUST",
    "DAMAGE",
    "DAMAGED",
    "MISSING",
    "CHECK",
    "FOUND",
    "BROKEN",
    "CRUSHED",
    "SMASHED",
    "THROWN AWAY",
    "TEST",
    "AMAZON",
    "DOTCOM",
    "EBAY",
    "SOLD AS SET",
]

pattern = "|".join(re.escape(term) for term in known_suspicious_terms)
known_suspicious_mask = desc_standard.str.contains(pattern, regex=True)

print(
    "Rows matching known suspicious Description terms: "
    f"{known_suspicious_mask.sum():,}"
)
print(
    "Unique matching descriptions: "
    f"{desc_text[known_suspicious_mask].nunique():,}"
)

print("\nTop matching descriptions:")
print(desc_text[known_suspicious_mask].value_counts().head(50))

Rows matching known suspicious Description terms: 5,254
Unique matching descriptions: 158

Top matching descriptions:
Description
DOTCOM POSTAGE                         1444
Manual                                 1426
BROWN CHECK CAT DOORSTOP                472
CARRIAGE                                279
Discount                                177
check                                   162
SUNSET CHECK HAMMOCK                    115
PAIR PADDED HANGERS PINK CHECK          100
Bank Charges                             96
damages                                  84
damaged                                  81
Next Day Carriage                        80
FRENCH CARRIAGE LANTERN                  63
BLACK BAROQUE CARRIAGE CLOCK             53
AMAZON FEE                               43
Adjustment by john on 26/01/2010 16      38
found                                    28
missing                                  27
Dotcomgiftshop Gift Voucher £20.00       26
Adjustment by john on 26/01/2010 1

In [28]:
# Description audit — missing Description cross-column profile
description_missing = df["Description"].isna()
missing_description_rows = df.loc[description_missing].copy()

category_lookup = non_conf_df.set_index("StockCode")["category"]
missing_description_rows["stockcode_category"] = (
    missing_description_rows["StockCode"]
    .map(category_lookup)
    .fillna("conforming product-code pattern")
)

print(f"Rows with missing Description: {len(missing_description_rows):,}")
print(
    "  with missing Customer ID: "
    f"{missing_description_rows['Customer ID'].isna().sum():,}"
)
print(
    "  with valid Customer ID:   "
    f"{missing_description_rows['Customer ID'].notna().sum():,}"
)

print("\nStockCode category breakdown for missing Description rows:")
print(
    missing_description_rows["stockcode_category"]
    .value_counts()
)

print("\nTop StockCodes with missing Description:")
print(
    missing_description_rows["StockCode"]
    .value_counts()
    .head(25)
)

Rows with missing Description: 4,382
  with missing Customer ID: 4,382
  with valid Customer ID:   0

StockCode category breakdown for missing Description rows:
stockcode_category
conforming product-code pattern    4315
DCGS-prefix code                     22
gift voucher code                    21
letters only                         14
single letter + digits                4
two-letter product-code variant       4
other                                 2
Name: count, dtype: int64

Top StockCodes with missing Description:
StockCode
22139     12
84990     12
79321     11
35965     11
22950     10
23084     10
22087      9
22084      9
71477      8
37461      8
72803B     7
37509      7
35970      7
84977      7
21768      7
22528      7
POST       7
84795D     7
22451      7
37446      6
21478      6
21169      6
21340      6
84845C     6
20747      6
Name: count, dtype: int64


**Description audit summary**

`Description` has missing values and clear whitespace inconsistencies, but all missing-description rows also have missing `Customer ID`. Casing and keyword scans are useful for investigation but not reliable enough as removal rules because they match both non-product rows and valid products.

For Stage 2, standardise `Description` whitespace. Row removal should rely mainly on stronger fields such as `StockCode`, `Customer ID`, `Quantity`, `Price`, and invoice cancellation patterns.


### Quantity

In [29]:
# Quantity audit — basic numeric profile
print(f"Dtype:         {df['Quantity'].dtype}")
print(f"Null count:    {df['Quantity'].isna().sum():,}")
print(f"Unique values: {df['Quantity'].nunique():,}")

print("\nSummary statistics:")
print(df["Quantity"].describe())

print("\nQuantity sign breakdown:")
print(f"Positive quantities: {(df['Quantity'] > 0).sum():,}")
print(f"Zero quantities:     {(df['Quantity'] == 0).sum():,}")
print(f"Negative quantities: {(df['Quantity'] < 0).sum():,}")

Dtype:         int64
Null count:    0
Unique values: 1,057

Summary statistics:
count    1.067371e+06
mean     9.938898e+00
std      1.727058e+02
min     -8.099500e+04
25%      1.000000e+00
50%      3.000000e+00
75%      1.000000e+01
max      8.099500e+04
Name: Quantity, dtype: float64

Quantity sign breakdown:
Positive quantities: 1,044,421
Zero quantities:     0
Negative quantities: 22,950


In [30]:
# Quantity audit — check negative quantities against cancellation invoices
invoice_text = df["Invoice"].astype(str).str.strip()

negative_quantity = df["Quantity"] < 0
cancel_invoice = invoice_text.str.startswith("C")

print(f"Negative Quantity rows: {negative_quantity.sum():,}")
print(f"  with cancellation invoice prefix:    {(negative_quantity & cancel_invoice).sum():,}")
print(f"  without cancellation invoice prefix: {(negative_quantity & ~cancel_invoice).sum():,}")

print("\nCustomer ID profile for negative Quantity rows:")
print(f"  missing Customer ID: {df.loc[negative_quantity, 'Customer ID'].isna().sum():,}")
print(f"  valid Customer ID:   {df.loc[negative_quantity, 'Customer ID'].notna().sum():,}")

print("\nTop StockCodes among negative Quantity rows:")
print(df.loc[negative_quantity, "StockCode"].value_counts().head(20))

print("\nTop Descriptions among negative Quantity rows:")
print(df.loc[negative_quantity, "Description"].value_counts().head(20))

Negative Quantity rows: 22,950
  with cancellation invoice prefix:    19,493
  without cancellation invoice prefix: 3,457

Customer ID profile for negative Quantity rows:
  missing Customer ID: 4,206
  valid Customer ID:   18,744

Top StockCodes among negative Quantity rows:
StockCode
M         537
22423     359
POST      229
22138     213
21232     190
D         172
21843     163
85123A    137
79323W    128
21527     122
S         101
22197      96
82483      91
22960      91
79323P     88
20914      88
85099B     87
20725      80
22720      76
21231      75
Name: count, dtype: int64

Top Descriptions among negative Quantity rows:
Description
Manual                                537
REGENCY CAKESTAND 3 TIER              347
POSTAGE                               229
BAKING SET 9 PIECE RETROSPOT          211
STRAWBERRY CERAMIC TRINKET BOX        184
Discount                              172
WHITE HANGING HEART T-LIGHT HOLDER    135
check                                 123
WHITE CHERRY

In [31]:
# Quantity audit — inspect negative quantities without cancellation invoice prefix
invoice_text = df["Invoice"].astype(str).str.strip()

negative_quantity = df["Quantity"] < 0
cancel_invoice = invoice_text.str.startswith("C")

negative_non_cancel = df.loc[negative_quantity & ~cancel_invoice].copy()

print(f"Negative Quantity rows without cancellation prefix: {len(negative_non_cancel):,}")

print("\nCustomer ID profile:")
print(f"  missing Customer ID: {negative_non_cancel['Customer ID'].isna().sum():,}")
print(f"  valid Customer ID:   {negative_non_cancel['Customer ID'].notna().sum():,}")

print("\nPrice summary:")
print(negative_non_cancel["Price"].describe())

print("\nTop StockCodes:")
print(negative_non_cancel["StockCode"].value_counts().head(20))

print("\nTop Descriptions:")
print(negative_non_cancel["Description"].value_counts().head(20))

print("\nSample rows:")
print(
    negative_non_cancel[
        ["Invoice", "StockCode", "Description", "Quantity", "Price", "Customer ID", "Country"]
    ]
    .head(20)
    .to_string(index=False)
)

Negative Quantity rows without cancellation prefix: 3,457

Customer ID profile:
  missing Customer ID: 3,457
  valid Customer ID:   0

Price summary:
count    3457.0
mean        0.0
std         0.0
min         0.0
25%         0.0
50%         0.0
75%         0.0
max         0.0
Name: Price, dtype: float64

Top StockCodes:
StockCode
22423     12
46000M     6
82494L     6
22719      6
46000S     5
84016      5
85017A     5
47566B     5
20852      5
85175      5
21830      5
71477      4
21768      4
84559D     4
84990      4
22028      4
35972      4
21040      4
79000      4
20713      4
Name: count, dtype: int64

Top Descriptions:
Description
check                     123
damages                    84
?                          83
damaged                    78
missing                    27
sold as set on dotcom      20
Damaged                    17
smashed                     9
thrown away                 9
Unsaleable, destroyed.      9
dotcom                      8
damages?            

In [32]:
# Quantity audit — inspect extreme quantity values
quantity_extremes = df.loc[
    df["Quantity"].abs().sort_values(ascending=False).head(25).index,
    ["Invoice", "StockCode", "Description", "Quantity", "Price", "Customer ID", "Country"]
]

print(quantity_extremes.to_string(index=False))

Invoice StockCode                         Description  Quantity  Price  Customer ID        Country
C581484     23843         PAPER CRAFT , LITTLE BIRDIE    -80995   2.08      16446.0 United Kingdom
 581483     23843         PAPER CRAFT , LITTLE BIRDIE     80995   2.08      16446.0 United Kingdom
 541431     23166      MEDIUM CERAMIC TOP STORAGE JAR     74215   1.04      12346.0 United Kingdom
C541433     23166      MEDIUM CERAMIC TOP STORAGE JAR    -74215   1.04      12346.0 United Kingdom
 497946     37410  BLACK AND WHITE PAISLEY FLOWER MUG     19152   0.10      13902.0        Denmark
 501534     21091         SET/6 WOODLAND PAPER PLATES     12960   0.10      13902.0        Denmark
 501534     21099         SET/6 STRAWBERRY PAPER CUPS     12960   0.10      13902.0        Denmark
 501534     21085           SET/6 WOODLAND PAPER CUPS     12744   0.10      13902.0        Denmark
 578841     84826      ASSTD DESIGN 3D PAPER STICKERS     12540   0.00      13256.0 United Kingdom
 501534   

**Quantity audit summary**

`Quantity` is numeric and has no missing or zero values. There are 22,950 negative rows: 19,493 have the `C` cancellation invoice prefix, while 3,457 do not.

All 3,457 non-cancellation negative rows have missing `Customer ID` and `Price = 0`, with descriptions such as `check`, `damages`, `missing`, and `thrown away`. These rows appear to be operational adjustment or write-off records rather than customer purchases.

Extreme quantities are mixed, including large customer purchases, cancellations, and zero-price operational records. Stage 2 should not use a simple outlier threshold; cancellation invoice removal and the `Quantity <= 0` filter cover the negative rows, while the missing-customer and non-positive-price filters cover the operational-adjustment rows surfaced here.

### InvoiceDate

In [33]:
# InvoiceDate audit — basic datetime profile
print(f"Dtype:         {df['InvoiceDate'].dtype}")
print(f"Null count:    {df['InvoiceDate'].isna().sum():,}")
print(f"Unique values: {df['InvoiceDate'].nunique():,}")

print("\nDate range:")
print(f"Earliest date: {df['InvoiceDate'].min()}")
print(f"Latest date:   {df['InvoiceDate'].max()}")

print("\nRows by year:")
print(df["InvoiceDate"].dt.year.value_counts().sort_index())

Dtype:         datetime64[us]
Null count:    0
Unique values: 47,635

Date range:
Earliest date: 2009-12-01 07:45:00
Latest date:   2011-12-09 12:50:00

Rows by year:
InvoiceDate
2009     45228
2010    522714
2011    499429
Name: count, dtype: int64


In [34]:
# InvoiceDate audit — monthly transaction distribution
monthly_rows = (
    df
    .assign(invoice_month=df["InvoiceDate"].dt.to_period("M"))
    .groupby("invoice_month")
    .size()
)

print(monthly_rows)

invoice_month
2009-12    45228
2010-01    31555
2010-02    29388
2010-03    41511
2010-04    34057
2010-05    35323
2010-06    39983
2010-07    33383
2010-08    33306
2010-09    42091
2010-10    59098
2010-11    78015
2010-12    65004
2011-01    35147
2011-02    27707
2011-03    36748
2011-04    29916
2011-05    37030
2011-06    36874
2011-07    39518
2011-08    35284
2011-09    50226
2011-10    60742
2011-11    84711
2011-12    25526
Freq: M, dtype: int64


In [35]:
# InvoiceDate audit — check date coverage at dataset boundaries
month_coverage = (
    df
    .assign(invoice_month=df["InvoiceDate"].dt.to_period("M"))
    .groupby("invoice_month")
    .agg(
        first_date=("InvoiceDate", "min"),
        last_date=("InvoiceDate", "max"),
        rows=("InvoiceDate", "size"),
    )
)

print("First three months:")
print(month_coverage.head(3).to_string())

print("\nLast three months:")
print(month_coverage.tail(3).to_string())

First three months:
                       first_date           last_date   rows
invoice_month                                               
2009-12       2009-12-01 07:45:00 2009-12-23 16:58:00  45228
2010-01       2010-01-04 09:24:00 2010-01-31 16:02:00  31555
2010-02       2010-02-01 08:13:00 2010-02-28 16:16:00  29388

Last three months:
                       first_date           last_date   rows
invoice_month                                               
2011-10       2011-10-02 10:32:00 2011-10-31 17:19:00  60742
2011-11       2011-11-01 08:16:00 2011-11-30 17:42:00  84711
2011-12       2011-12-01 08:12:00 2011-12-09 12:50:00  25526


**InvoiceDate audit summary**

`InvoiceDate` is stored as a datetime column and has no missing values. The dataset covers 1 December 2009 to 9 December 2011.

Both boundary months are incomplete: 2009-12 starts on 1 December and ends on 23 December, while 2011-12 ends on 9 December. Monthly volume is highest in October and November in both full years.

Stage 2 cohort, retention, and churn windows should handle boundary months explicitly by either excluding them or documenting the truncation.

### Price

In [36]:
# Price audit — basic numeric profile
print(f"Dtype:         {df['Price'].dtype}")
print(f"Null count:    {df['Price'].isna().sum():,}")
print(f"Unique values: {df['Price'].nunique():,}")

print("\nSummary statistics:")
print(df["Price"].describe())

print("\nPrice sign breakdown:")
print(f"Positive prices: {(df['Price'] > 0).sum():,}")
print(f"Zero prices:     {(df['Price'] == 0).sum():,}")
print(f"Negative prices: {(df['Price'] < 0).sum():,}")

Dtype:         float64
Null count:    0
Unique values: 2,807

Summary statistics:
count    1.067371e+06
mean     4.649388e+00
std      1.235531e+02
min     -5.359436e+04
25%      1.250000e+00
50%      2.100000e+00
75%      4.150000e+00
max      3.897000e+04
Name: Price, dtype: float64

Price sign breakdown:
Positive prices: 1,061,164
Zero prices:     6,202
Negative prices: 5


In [37]:
# Price audit — inspect negative prices
negative_price_rows = df.loc[df["Price"] < 0]

print(f"Negative Price rows: {len(negative_price_rows):,}")

print(
    negative_price_rows[
        ["Invoice", "StockCode", "Description", "Quantity", "Price", "Customer ID", "Country"]
    ]
    .to_string(index=False)
)

Negative Price rows: 5
Invoice StockCode     Description  Quantity     Price  Customer ID        Country
A506401         B Adjust bad debt         1 -53594.36          NaN United Kingdom
A516228         B Adjust bad debt         1 -44031.79          NaN United Kingdom
A528059         B Adjust bad debt         1 -38925.87          NaN United Kingdom
A563186         B Adjust bad debt         1 -11062.06          NaN United Kingdom
A563187         B Adjust bad debt         1 -11062.06          NaN United Kingdom


In [38]:
# Price audit — inspect zero-price rows
zero_price_rows = df.loc[df["Price"] == 0]

print(f"Zero Price rows: {len(zero_price_rows):,}")

print("\nCustomer ID profile:")
print(f"  missing Customer ID: {zero_price_rows['Customer ID'].isna().sum():,}")
print(f"  valid Customer ID:   {zero_price_rows['Customer ID'].notna().sum():,}")

print("\nQuantity sign breakdown:")
print(f"  positive Quantity: {(zero_price_rows['Quantity'] > 0).sum():,}")
print(f"  negative Quantity: {(zero_price_rows['Quantity'] < 0).sum():,}")
print(f"  zero Quantity:     {(zero_price_rows['Quantity'] == 0).sum():,}")

print("\nTop StockCodes:")
print(zero_price_rows["StockCode"].value_counts().head(20))

print("\nTop Descriptions:")
print(zero_price_rows["Description"].value_counts().head(20))

Zero Price rows: 6,202

Customer ID profile:
  missing Customer ID: 6,131
  valid Customer ID:   71

Quantity sign breakdown:
  positive Quantity: 2,745
  negative Quantity: 3,457
  zero Quantity:     0

Top StockCodes:
StockCode
46000M    18
22501     18
79321     17
21116     16
22423     16
23084     16
46000S    15
22734     15
22139     14
35965     14
84990     13
22502     13
20713     13
22469     12
84016     11
22627     11
71477     10
22470     10
22625     10
22624     10
Name: count, dtype: int64

Top Descriptions:
Description
check                                  162
?                                       92
damages                                 84
damaged                                 81
found                                   28
missing                                 27
sold as set on dotcom                   20
Damaged                                 17
adjustment                              16
OWL DOORSTOP                            15
POLYESTER FILLER PAD 45

In [39]:
# Price audit — inspect zero-price rows with valid Customer ID
zero_price_valid_customer = df.loc[
    (df["Price"] == 0) & df["Customer ID"].notna()
]

print(f"Zero Price rows with valid Customer ID: {len(zero_price_valid_customer):,}")

print("\nQuantity profile:")
print(zero_price_valid_customer["Quantity"].describe())

print("\nTop StockCodes:")
print(zero_price_valid_customer["StockCode"].value_counts().head(20))

print("\nTop Descriptions:")
print(zero_price_valid_customer["Description"].value_counts().head(20))

print("\nSample rows:")
print(
    zero_price_valid_customer[
        ["Invoice", "StockCode", "Description", "Quantity", "Price", "Customer ID", "Country"]
    ]
    .head(25)
    .to_string(index=False)
)

Zero Price rows with valid Customer ID: 71

Quantity profile:
count       71.000000
mean       207.802817
std       1487.153922
min          1.000000
25%          1.000000
50%          5.000000
75%         12.000000
max      12540.000000
Name: Quantity, dtype: float64

Top StockCodes:
StockCode
M          7
22065      2
TEST001    2
22423      2
22841      2
22076      1
48185      1
22142      1
85042      1
21143      1
79320      1
22355      1
21533      1
21662      1
22459      1
22458      1
22376      1
21765      1
20914      1
22690      1
Name: count, dtype: int64

Top Descriptions:
Description
Manual                               7
CHRISTMAS PUDDING TRINKET POT        2
This is a test product.              2
REGENCY CAKESTAND 3 TIER             2
ROUND CAKE TIN VINTAGE GREEN         2
6 RIBBONS EMPIRE                     1
DOOR MAT FAIRY CAKE                  1
CHRISTMAS CRAFT WHITE FAIRY          1
ANTIQUE LILY FAIRY LIGHTS            1
ANTIQUE GLASS HEART DECORATION      

**Price audit summary**

`Price` is numeric and has no missing values. 5 rows have negative unit prices, all `A`-prefix bad-debt adjustments with missing `Customer ID`, and 6,202 rows have zero unit prices. Of the zero-price rows, 6,131 have missing `Customer ID` and are covered by the missing-customer filter, while 71 are linked to valid customers.

These 71 rows have zero monetary value and would inflate RFM frequency without contributing to monetary value, so Stage 2 should remove all `Price <= 0` rows explicitly rather than relying on the missing-customer filter alone.

### Customer ID

In [40]:
# Customer ID audit — basic profile
print(f"Dtype:         {df['Customer ID'].dtype}")
print(f"Null count:    {df['Customer ID'].isna().sum():,}")
print(f"Null pct:      {df['Customer ID'].isna().mean():.2%}")
print(f"Unique values: {df['Customer ID'].nunique():,}")

print("\nSample Customer IDs:")
print(df["Customer ID"].dropna().head(10).tolist())

Dtype:         float64
Null count:    243,007
Null pct:      22.77%


Unique values: 5,942

Sample Customer IDs:
[13085.0, 13085.0, 13085.0, 13085.0, 13085.0, 13085.0, 13085.0, 13085.0, 13085.0, 13085.0]


In [41]:
# Customer ID audit — check valid ID structure
customer_ids = df["Customer ID"].dropna()

whole_number_ids = customer_ids.mod(1).eq(0)
positive_ids = customer_ids.gt(0)

print(f"Non-null Customer ID rows: {len(customer_ids):,}")
print(f"Whole-number IDs:          {whole_number_ids.sum():,}")
print(f"Non-whole-number IDs:      {(~whole_number_ids).sum():,}")
print(f"Positive IDs:              {positive_ids.sum():,}")
print(f"Non-positive IDs:          {(~positive_ids).sum():,}")

print("\nCustomer ID range:")
print(f"Minimum ID: {customer_ids.min():.0f}")
print(f"Maximum ID: {customer_ids.max():.0f}")

Non-null Customer ID rows: 824,364
Whole-number IDs:          824,364
Non-whole-number IDs:      0
Positive IDs:              824,364
Non-positive IDs:          0

Customer ID range:
Minimum ID: 12346
Maximum ID: 18287


In [42]:
# Customer ID audit — transaction volume per customer
customer_row_counts = df["Customer ID"].value_counts()

print("Rows per customer summary:")
print(customer_row_counts.describe())

print("\nTop 20 customers by row count:")
print(customer_row_counts.head(20))

Rows per customer summary:
count     5942.000000
mean       138.735106
std        359.689585
min          1.000000
25%         21.000000
50%         53.000000
75%        144.000000
max      13097.000000
Name: count, dtype: float64

Top 20 customers by row count:
Customer ID
17841.0    13097
14911.0    11613
12748.0     7307
14606.0     6709
14096.0     5128
15311.0     4717
14156.0     4130
14646.0     3890
13089.0     3438
16549.0     3255
14298.0     2868
14527.0     2837
17850.0     2827
15039.0     2810
15005.0     2548
13081.0     2430
17511.0     2134
13263.0     1920
16782.0     1900
14159.0     1885
Name: count, dtype: int64


**Customer ID audit summary**

`Customer ID` has 243,007 missing values (22.77%) and is the main data-completeness issue. The 824,364 non-null IDs are all positive whole numbers ranging from 12,346 to 18,287 across 5,942 unique customers, but pandas stores the column as `float64` because of the missing values.

Customer activity is uneven, with some customers appearing in thousands of transaction rows. This is expected in retail data but should be considered when building customer-level features.

For Stage 2, remove rows with missing `Customer ID`, then convert the column to a nullable `Int64` dtype.

### Country

In [43]:
# Country audit — basic profile
print(f"Dtype:         {df['Country'].dtype}")
print(f"Null count:    {df['Country'].isna().sum():,}")
print(f"Unique values: {df['Country'].nunique():,}")

print("\nTop countries by row count:")
print(df["Country"].value_counts().head(20))

print("\nBottom countries by row count:")
print(df["Country"].value_counts().tail(20))

Dtype:         str
Null count:    0
Unique values: 43

Top countries by row count:
Country
United Kingdom     981330
EIRE                17866
Germany             17624
France              14330
Netherlands          5140
Spain                3811
Switzerland          3189
Belgium              3123
Portugal             2620
Australia            1913
Channel Islands      1664
Italy                1534
Norway               1455
Sweden               1364
Cyprus               1176
Finland              1049
Austria               938
Denmark               817
Unspecified           756
Greece                663
Name: count, dtype: int64

Bottom countries by row count:
Country
United Arab Emirates    500
Israel                  371
Hong Kong               364
Singapore               346
Malta                   299
Iceland                 253
Canada                  228
Lithuania               189
RSA                     169
Bahrain                 126
Brazil                   94
Thailand       

In [44]:
# Country audit — check country distribution after Customer ID filter
customer_rows = df.loc[df["Customer ID"].notna()]

print(f"Rows with valid Customer ID: {len(customer_rows):,}")
print(f"Unique countries after Customer ID filter: {customer_rows['Country'].nunique():,}")

print("\nTop countries after Customer ID filter:")
print(customer_rows["Country"].value_counts().head(20))

print("\nCountries removed entirely by Customer ID filter:")
countries_raw = set(df["Country"].unique())
countries_with_customer = set(customer_rows["Country"].unique())
removed_countries = sorted(countries_raw - countries_with_customer)

print(removed_countries)

Rows with valid Customer ID: 824,364
Unique countries after Customer ID filter: 41

Top countries after Customer ID filter:
Country
United Kingdom     741301
Germany             17624
EIRE                16195
France              14202
Netherlands          5140
Spain                3811
Belgium              3123
Switzerland          3064
Portugal             2504
Australia            1913
Channel Islands      1664
Italy                1534
Norway               1455
Sweden               1345
Cyprus               1176
Finland              1049
Austria               938
Denmark               817
Greece                663
Japan                 582
Name: count, dtype: int64

Countries removed entirely by Customer ID filter:
['Bermuda', 'Hong Kong']


**Country audit summary**

`Country` has no missing values and is heavily dominated by the United Kingdom. After filtering to rows with valid `Customer ID`, most countries remain, but countries with no customer identifiers are removed.

For Stage 2, `Country` can be kept as a customer and transaction feature. Later analysis should account for the strong UK dominance when comparing countries or building customer-level segments.

### Duplicate rows

In [45]:
# Duplicate-row audit — check exact full-row duplicates
duplicate_rows = df.duplicated()
duplicate_group_rows = df.duplicated(keep=False)

print(f"Exact duplicate rows:               {duplicate_rows.sum():,}")
print(f"Rows belonging to duplicate groups: {duplicate_group_rows.sum():,}")

print("\nSample duplicate rows:")
print(
    df.loc[duplicate_group_rows, INSPECTION_COLUMNS]
    .head(20)
    .to_string(index=False)
)

Exact duplicate rows:               34,335
Rows belonging to duplicate groups: 67,242

Sample duplicate rows:
Invoice StockCode                       Description  Quantity         InvoiceDate  Price  Customer ID        Country
 489517     21913    VINTAGE SEASIDE JIGSAW PUZZLES         1 2009-12-01 11:34:00   3.75      16329.0 United Kingdom
 489517     21912          VINTAGE SNAKES & LADDERS         1 2009-12-01 11:34:00   3.75      16329.0 United Kingdom
 489517     21821  GLITTER STAR GARLAND WITH BELLS          1 2009-12-01 11:34:00   3.75      16329.0 United Kingdom
 489517     22319 HAIRCLIPS FORTIES FABRIC ASSORTED        12 2009-12-01 11:34:00   0.65      16329.0 United Kingdom
 489517     22130  PARTY CONE CHRISTMAS DECORATION          6 2009-12-01 11:34:00   0.85      16329.0 United Kingdom
 489517     21912          VINTAGE SNAKES & LADDERS         1 2009-12-01 11:34:00   3.75      16329.0 United Kingdom
 489517     21491   SET OF THREE VINTAGE GIFT WRAPS         1 2009-12-0

**Duplicate rows summary**

The dataset contains 34,335 exact duplicate rows. In total, 67,242 rows belong to duplicate groups when including the original rows. If left in the analysis, duplicate records can inflate transaction counts, revenue, and RFM metrics.

Stage 2 should remove exact full-row duplicates before feature engineering.

## Final Stage 1 summary and cleaning plan

The raw dataset contains 1,067,371 rows and 8 columns from two yearly sheets. The main cleaning risks are missing customer identifiers, cancellation records, non-product StockCodes, zero or negative prices, negative quantities, exact duplicate rows, and inconsistent text formatting.

Stage 2 cleaning should apply the following steps:

1. Create a working copy of the raw data.
2. Standardise text fields for matching: ensure `Invoice` and `StockCode` are string-typed, strip whitespace from `StockCode` and `Description`, and collapse repeated spaces in `Description`.
3. Remove rows with missing `Customer ID`.
4. Remove rows where `Invoice` does not match the 6-digit standard pattern, covering cancellation invoices and `A`-prefix bad-debt adjustments.
5. Remove rows with `Quantity <= 0`.
6. Remove rows with `Price <= 0`.
7. Remove the 17 non-product StockCodes identified in the StockCode audit: `POST`, `DOT`, `M`, `m`, `D`, `S`, `ADJUST`, `AMAZONFEE`, `CRUK`, `B`, `GIFT`, `PADS`, `BANK CHARGES`, `C2`, `TEST001`, `TEST002`, and `ADJUST2`.
8. Remove exact full-row duplicates with `df.drop_duplicates()`.
9. Convert `Customer ID` to a nullable `Int64` dtype.
10. Create `Revenue = Quantity * Price`.
11. Save the cleaned customer analytics dataset in `data/processed/`.

The cleaned dataset will support RFM segmentation, cohort retention analysis, churn modelling, and the Power BI dashboard.